# DharmaOCR — Notebook 01: Setup do dataset, figura de mérito e leitor

Primeiro notebook do POC de estudo do **Dharma OCR** e comparação com outros OCRs do mercado.

Aqui montamos a base que os próximos notebooks vão reutilizar:

1. **Download do dataset** `Dharma-AI/DharmaOCR-Benchmark` (Hugging Face).
2. **Figura de mérito** exatamente como no paper *DharmaOCR: Specialized Small Language Models for Structured OCR* (arXiv:2604.14314).
3. **Chamada ao leitor** (API HTTP do Dharma OCR) + um harness de avaliação genérico que aceita qualquer OCR (para plugar concorrentes no notebook seguinte).

## Figura de mérito (paper, Eq. 1)

$$\text{score} = \frac{\text{Levenshtein}_{Ratio} + \text{BLEU}}{2}$$

- **Levenshtein_Ratio**: distância de edição *normalizada* (∈ [0,1]) entre transcrição prevista e o ground‑truth — captura fidelidade caractere a caractere (acentos, pontuação, erros de grafia).
- **BLEU**: sobreposição de n‑gramas (∈ [0,1] após dividir por 100) — mais sensível à preservação de ordem/sequência de tokens.
- O score do documento é a média dos dois; o score do modelo é a **média dos scores por documento**.

Métricas operacionais secundárias que o paper também acompanha: **text degeneration rate** (requisições que estouram o limite de tokens **ou** exibem repetição de n‑gramas) e **custo por página**.

## Dataset (paper, Seção 5.1)

`Dharma-AI/DharmaOCR-Benchmark` — split `test`, 496 instâncias em PT‑BR: **ESTER‑Pt** (363), **Legal** (83), **BRESSAY** (50, manuscrito). Colunas:

| coluna | conteúdo |
|---|---|
| `id` | identificador inteiro |
| `image` | imagem PNG da página (PIL) |
| `image_base64` | a mesma imagem em base64 (entrada para APIs) |
| `assistant` | ground‑truth em **JSON estrito** `{header, text, footer, margin}` |
| `assistant_without_json` | transcrição em **texto puro** (ground‑truth de texto) |

> Referências: Benchmark `Dharma-AI/DharmaOCR-Benchmark` · Modelo `Dharma-AI/DharmaOCR-Lite` · site `dharma-ai.com.br`.


## 1. Dependências

Ambiente alvo: **Python 3.8**. As versões abaixo são fixadas para compatibilidade.
`rapidfuzz`, `numpy`, `pandas`, `requests` e `Pillow` normalmente já estão presentes; `datasets`, `huggingface_hub` e `sacrebleu` costumam faltar.

Rode a célula abaixo uma vez (descomente se precisar instalar).

In [ ]:
# Colab: instala só o que falta (sem "-U" em libs core do Colab).
%pip install -q datasets rapidfuzz sacrebleu
# (Local Python 3.8: use os pins  "datasets<3.0" "huggingface_hub<0.24" "sacrebleu>=2.3,<3".)
print("deps prontas.")

In [ ]:
import os, re, json, time, base64, io
from dataclasses import dataclass, field
from typing import Callable, Optional, List, Dict, Any

import numpy as np
import pandas as pd

# Métrica
from rapidfuzz.distance import Levenshtein as RFLevenshtein
import sacrebleu

print("sacrebleu:", sacrebleu.__version__)
print("imports OK")

## 2. Download do dataset

Baixamos o split `test` do benchmark. As imagens vêm como PIL em `image`; a versão base64 (entrada para a API) está em `image_base64`.

In [ ]:
from datasets import load_dataset

DATASET_ID = "Dharma-AI/DharmaOCR-Benchmark"

ds = load_dataset(DATASET_ID, split="test")
print(ds)
print("n =", len(ds))
print("colunas:", ds.column_names)

In [ ]:
# Espia uma amostra (sem despejar o base64 inteiro no output)
ex = ds[0]
print("id:", ex["id"])
print("image:", ex["image"].size, ex["image"].mode)
print("image_base64 (len):", len(ex["image_base64"]))
print("\n--- assistant (ground-truth JSON, 400 chars) ---")
print(ex["assistant"][:400])
print("\n--- assistant_without_json (ground-truth texto, 400 chars) ---")
print(ex["assistant_without_json"][:400])

In [ ]:
# Visualiza a página da amostra
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 9))
plt.imshow(ex["image"])
plt.axis("off")
plt.title(f"DharmaOCR-Benchmark · id={ex['id']}")
plt.show()

## 3. Figura de mérito

Implementação fiel à Eq. 1 do paper.

- `levenshtein_ratio(pred, ref)` → similaridade de edição normalizada (rapidfuzz, ∈ [0,1]).
- `bleu(pred, ref)` → BLEU sentença a sentença via sacrebleu, dividido por 100 (∈ [0,1]).
- `dharma_score(pred, ref)` → média dos dois.
- Agregação: `mean` dos scores por documento.

**Normalização.** O paper não aplica normalização pesada (o Levenshtein já penaliza acento/pontuação). Deixamos `normalize=False` como padrão para replicar o paper; há um `normalize_text` opcional (colapsa espaços) para comparações mais tolerantes entre OCRs de mercado.

**Referência de texto.** O ground‑truth textual é `assistant_without_json`. Como cada OCR de mercado devolve texto puro (não o JSON do Dharma), comparar contra `assistant_without_json` dá um terreno comum justo. Para uma replicação estrita do modelo Dharma (que emite o JSON), basta trocar a referência para `assistant` e extrair o texto do JSON com `transcription_from_output`.

In [ ]:
def normalize_text(s: str) -> str:
    """Normalização leve e opcional: colapsa espaços/quebras e apara as pontas."""
    if s is None:
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()


def levenshtein_ratio(pred: str, ref: str) -> float:
    """Distância de Levenshtein normalizada em [0,1] (1.0 = idêntico)."""
    pred = "" if pred is None else str(pred)
    ref = "" if ref is None else str(ref)
    if not pred and not ref:
        return 1.0
    return float(RFLevenshtein.normalized_similarity(pred, ref))


def bleu(pred: str, ref: str) -> float:
    """BLEU (sacrebleu, tokenizador padrão '13a') em [0,1]."""
    pred = "" if pred is None else str(pred)
    ref = "" if ref is None else str(ref)
    if not pred.strip() or not ref.strip():
        return 0.0
    return sacrebleu.sentence_bleu(pred, [ref]).score / 100.0


def dharma_score(pred: str, ref: str, normalize: bool = False) -> Dict[str, float]:
    """Score do paper: (Levenshtein_Ratio + BLEU) / 2, mais os componentes."""
    if normalize:
        pred, ref = normalize_text(pred), normalize_text(ref)
    lev = levenshtein_ratio(pred, ref)
    bl = bleu(pred, ref)
    return {"levenshtein_ratio": lev, "bleu": bl, "score": (lev + bl) / 2.0}

In [ ]:
# Sanity check da métrica
assert dharma_score("abc", "abc")["score"] == 1.0
d_ident = dharma_score("O rato roeu a roupa do rei de Roma.", "O rato roeu a roupa do rei de Roma.")
d_diff  = dharma_score("xxxxx yyyyy zzzzz", "O rato roeu a roupa do rei de Roma.")
d_near  = dharma_score("O rato roeu a roupa do rei de Romo.", "O rato roeu a roupa do rei de Roma.")
print("idêntico:", d_ident)
print("quase   :", d_near)
print("diferente:", d_diff)
assert d_ident["score"] > d_near["score"] > d_diff["score"]
print("\nOK: métrica ordena idêntico > quase > diferente.")

### 3.1 Text degeneration rate (métrica secundária)

O paper marca uma requisição como *degenerada* quando ela **(i)** atinge o limite de tokens de saída **ou** **(ii)** exibe repetição de n‑gramas. Implementamos o critério de repetição; o estouro de token limit depende do metadado que a API devolver (`finish_reason`/`truncated`), passado via `hit_token_limit`.

In [ ]:
def has_ngram_repetition(text: str, n: int = 4, max_repeats: int = 3) -> bool:
    """True se algum n-grama (em palavras) se repete mais que `max_repeats` vezes."""
    toks = normalize_text(text).split()
    if len(toks) < n:
        return False
    counts: Dict[str, int] = {}
    for i in range(len(toks) - n + 1):
        g = " ".join(toks[i:i + n])
        counts[g] = counts.get(g, 0) + 1
        if counts[g] > max_repeats:
            return True
    return False


def is_degenerate(text: str, hit_token_limit: bool = False,
                  n: int = 4, max_repeats: int = 3) -> bool:
    return bool(hit_token_limit) or has_ngram_repetition(text, n=n, max_repeats=max_repeats)


# check rápido
assert has_ngram_repetition("a b c d " * 6) is True
assert has_ngram_repetition("O rato roeu a roupa do rei de Roma") is False
print("OK: detector de degeneração.")

## 4. O leitor (Dharma OCR — API HTTP real)

Contrato: **Smart OCR API v2.0.0**, endpoint síncrono `POST /v1/ocrs/` (tag *OCR Sync v1*).

- **Base URL**: `https://ocr-api.com-us-east-2.dharma-ai.com` → `DHARMA_API_URL` já aponta para `/v1/ocrs/`.
- **Auth**: `Authorization: Bearer <API key>` — setar em `DHARMA_API_KEY` (via `os.environ`, **não commitar**).
- **Request** (`ExecuteOCRRequestDTO`): `b64image` (data URI `data:image/png;base64,...`, **obrigatório**), `model` (`full` padrão | `lite`); opcionais `lora`, `response_format`, `filename`, `postprocess_config`.
- **Response** (`OCRResponseDTO`): `text` = transcrição da página (é o que entra na métrica), `metadata` com `truncated_pages` (→ `hit_token_limit` do *degeneration rate*), `num_output_tokens`, `total_time`.

> Para OCR de texto puro (a comparação do paper) usamos só `b64image` + `model`, sem `lora`/`response_format`, e pontuamos `response.text` contra `assistant_without_json`. O modelo do paper (`full`/`lite`) é o próprio parâmetro `model`.


In [ ]:
import requests

# Dharma OCR — API HTTP real (Smart OCR API v2.0.0)
DHARMA_API_URL = os.environ.get("DHARMA_API_URL", "https://ocr-api.com-us-east-2.dharma-ai.com/v1/ocrs/")
DHARMA_API_KEY = os.environ.get("DHARMA_API_KEY", "")     # <-- sua API key (Bearer). NÃO comitar.
DHARMA_MODEL   = os.environ.get("DHARMA_MODEL", "full")   # "full" (máx. acurácia) ou "lite" (mais rápido)
DHARMA_TIMEOUT = int(os.environ.get("DHARMA_TIMEOUT", "180"))


@dataclass
class OCRResult:
    text: str                       # transcrição em texto puro (entra na métrica)
    raw: Any = None                 # resposta crua do provedor
    hit_token_limit: bool = False   # p/ degeneration rate (metadata.truncated_pages)
    latency_s: float = 0.0
    out_tokens: Optional[int] = None
    error: Optional[str] = None


# Assinatura de um leitor: recebe image_base64 (base64 cru OU data URI) -> OCRResult
OCRReader = Callable[[str], OCRResult]


def to_data_uri(image_base64: str, mime: str = "image/png") -> str:
    """A API exige data URI (`data:image/png;base64,...`). O dataset traz base64 cru."""
    s = image_base64 or ""
    return s if s.startswith("data:") else f"data:{mime};base64,{s}"


def transcription_from_output(obj: Any) -> str:
    """Extrai texto puro de JSON {header,text,footer,margin}, dict ou string.
    Usado p/ o ground-truth `assistant` (replicação estrita) — a resposta da API já vem em `text`."""
    if obj is None:
        return ""
    if isinstance(obj, str):
        s = obj.strip()
        if s.startswith("{"):
            try:
                obj = json.loads(s)
            except json.JSONDecodeError:
                return obj
        else:
            return obj
    if isinstance(obj, dict):
        parts = [obj.get(k, "") for k in ("header", "text", "footer", "margin")]
        parts = [p for p in parts if p]
        return "\n".join(parts) if parts else json.dumps(obj, ensure_ascii=False)
    return str(obj)

print("Endpoint:", DHARMA_API_URL, "| model:", DHARMA_MODEL, "| key setada?", bool(DHARMA_API_KEY))

In [ ]:
def dharma_read(image_base64: str, model: Optional[str] = None) -> OCRResult:
    """POST /v1/ocrs/ (Smart OCR API v1, síncrono). Devolve a transcrição em `text`.

    Body:  {"b64image": "data:image/png;base64,...", "model": "full"|"lite"}
    Auth:  Authorization: Bearer <API key>
    Resp.: {"id","filename","text",...,"metadata":{"truncated_pages":[...],"num_output_tokens":N,...}}
    """
    if not DHARMA_API_KEY:
        raise RuntimeError("Configure DHARMA_API_KEY (Bearer) antes de chamar dharma_read.")
    payload = {"b64image": to_data_uri(image_base64), "model": model or DHARMA_MODEL}
    headers = {"Authorization": f"Bearer {DHARMA_API_KEY}", "Content-Type": "application/json"}
    t0 = time.time()
    try:
        resp = requests.post(DHARMA_API_URL, json=payload, headers=headers, timeout=DHARMA_TIMEOUT)
        dt = time.time() - t0
        resp.raise_for_status()
        data = resp.json()
        text = data.get("text", "") or ""
        meta = data.get("metadata") or {}
        truncated = bool(meta.get("truncated_pages"))
        return OCRResult(text=text, raw=data, hit_token_limit=truncated,
                         latency_s=dt, out_tokens=meta.get("num_output_tokens"))
    except Exception as e:  # noqa: BLE001 - registra a falha por-doc e segue a avaliação
        return OCRResult(text="", raw=None, latency_s=time.time() - t0, error=str(e))


# Leitor "oráculo" (copia o ground-truth) p/ exercitar o harness sem gastar créditos da API.
def make_oracle_reader(dataset) -> OCRReader:
    ref_by_pos = {i: dataset[i]["assistant_without_json"] for i in range(len(dataset))}
    def _reader(image_base64: str, _pos=[0]) -> OCRResult:
        i = _pos[0]; _pos[0] += 1
        return OCRResult(text=ref_by_pos.get(i, ""))
    return _reader

print("leitor Dharma pronto (POST /v1/ocrs/). Configurado?", bool(DHARMA_API_KEY))

## 5. Harness de avaliação

`evaluate_reader` roda um leitor sobre N amostras, calcula o score por documento e agrega. Reutilizável para o Dharma e para qualquer OCR concorrente (basta passar outra função `OCRReader`).

O ground‑truth de texto padrão é `assistant_without_json` (`reference_col`).

In [ ]:
def evaluate_reader(reader: OCRReader,
                    dataset,
                    n: Optional[int] = None,
                    reference_col: str = "assistant_without_json",
                    normalize: bool = False,
                    sleep_s: float = 0.0,
                    verbose: bool = True) -> pd.DataFrame:
    """Avalia `reader` e devolve um DataFrame por documento (id, score, componentes, degen, latência, erro)."""
    n = len(dataset) if n is None else min(n, len(dataset))
    rows = []
    for i in range(n):
        ex = dataset[i]
        ref = ex[reference_col]
        res = reader(ex["image_base64"])
        sc = dharma_score(res.text, ref, normalize=normalize)
        rows.append({
            "id": ex["id"],
            "score": sc["score"],
            "levenshtein_ratio": sc["levenshtein_ratio"],
            "bleu": sc["bleu"],
            "degenerate": is_degenerate(res.text, res.hit_token_limit),
            "latency_s": round(res.latency_s, 3),
            "error": res.error,
            "pred_len": len(res.text or ""),
            "ref_len": len(ref or ""),
        })
        if verbose and (i + 1) % 25 == 0:
            print(f"  {i+1}/{n} ...")
        if sleep_s:
            time.sleep(sleep_s)
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> Dict[str, float]:
    """Agrega no formato do paper: score médio, componentes médios, degeneration rate, erros."""
    ok = df[df["error"].isna()]
    return {
        "n": int(len(df)),
        "n_ok": int(len(ok)),
        "n_error": int(df["error"].notna().sum()),
        "score": float(ok["score"].mean()) if len(ok) else float("nan"),
        "levenshtein_ratio": float(ok["levenshtein_ratio"].mean()) if len(ok) else float("nan"),
        "bleu": float(ok["bleu"].mean()) if len(ok) else float("nan"),
        "degeneration_rate_pct": float(100.0 * df["degenerate"].mean()),
        "latency_s_median": float(df["latency_s"].median()),
    }

### 5.1 Smoke test

Sem a API configurada, exercitamos o harness com o **leitor-oráculo** (copia o ground‑truth → score ≈ 1.0) só para validar a mecânica. Assim que `DHARMA_API_URL`/`DHARMA_API_KEY` estiverem setados, troque `reader` por `dharma_read`.

In [ ]:
USE_REAL_DHARMA = bool(DHARMA_API_URL and DHARMA_API_KEY)

if USE_REAL_DHARMA:
    reader = dharma_read
    print("Usando a API real do Dharma OCR.")
else:
    reader = make_oracle_reader(ds)
    print("API não configurada -> usando leitor-oráculo (sanity check do harness).")

df = evaluate_reader(reader, ds, n=5, verbose=False)
display(df)
print("\nResumo:", json.dumps(summarize(df), ensure_ascii=False, indent=2))

## 6. Próximos passos

- **Notebook 02** — rodar o `dharma_read` no benchmark inteiro (496 docs) e reproduzir o número do paper para o Dharma (Lite ≈ 0.911 / Full ≈ 0.925).
- **Notebook 03** — plugar OCRs de mercado (Google Vision, AWS Textract, Azure/Document AI, GPT‑4o, Gemini, etc.) como funções `OCRReader` e comparar na **mesma** figura de mérito, quebrando por subconjunto (ESTER‑Pt / Legal / BRESSAY) e reportando também *degeneration rate*, latência e custo/página.

Tudo em `evaluate_reader(...) -> DataFrame` + `summarize(...)`, então a comparação entre OCRs é só um `concat` das tabelas.
